In [8]:
%pip install -U langchain-openai langchain-community langchain_classic langchain-text-splitters

In [7]:
from google.colab import userdata
api_key=userdata.get('OPENAI_API_KEY')

In [9]:
import os

import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from pathlib import Path
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.

제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

1. 개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

2. 주요 트렌드
- RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
- 멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
- AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
- 소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

3. 시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

1. 제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

2. 초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

3. 주요 기능
- 음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
- 자동 스케줄: 시간대별 기기 자동 제어
- 에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
- 보안 모드: 외출 시 자동 보안 설정
"""
}

SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

for filename, content in sample_texts.items():
    (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')

In [6]:
import os
import time
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

llm = ChatOpenAI(model='gpt-4o-mini')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
from pathlib import Path

data_dir = Path("sample_data")

files = {
    data_dir / "company_policy.txt" : '사내규정',
    data_dir / 'product_manual.txt' : '제품매뉴얼',
    data_dir / 'ai_report.txt' : 'AI보고서'
}

documents = []
for fpath, category in files.items():
    text = fpath.read_text()
    for section in text.strip().split('\n\n'):
        if section.strip():
            documents.append(Document(
                page_content = section.strip(),
                metadata = {'category' : category, 'source' : fpath.name}
            ))

In [ ]:
len(documents)

13

In [ ]:
documents[0]

Document(metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')

In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [ ]:
vectorstore.save_local('faiss_docs')

In [ ]:
loaded_vs = FAISS.load_local('faiss_docs', embeddings, allow_dangerous_deserialization=True)

In [ ]:
loaded_vs.index.ntotal

13

In [ ]:
retriever.invoke('재택근무 규정')

[Document(id='9355aa5d-922e-4318-b8c0-5407cf72633d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
 Document(id='59f4aea0-a1e6-42a1-9e17-906345867d24', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.'),
 Document(id='490be4d7-2fe3-406a-b6aa-18e7172e902f', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]

In [ ]:
# LCEL : LangChain Expression Language
#      | |  |

In [ ]:
# prompt | llm  | StrOutputParser()

In [ ]:
simple_prompt = ChatPromptTemplate.from_messages([
    ('user', '{question}')
])

simple_chain = simple_prompt | llm  | StrOutputParser()

In [ ]:
result = simple_chain.invoke({'question': '대한민국의 수도는?'})

In [ ]:
print(result)

대한민국의 수도는 서울입니다.


In [ ]:
# query -> retriever 검색을 해올거고   ->  llm

# llm  : ('재택근무 규정이 어떻게되나요?', 'retrieved documents')

# rag_chain = "query" | retriever | "결과" -> |  llm  | StrOutputParser()

In [ ]:
# RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

In [ ]:
passthrough.invoke('안녕하세요')

'안녕하세요'

In [ ]:
passthrough.invoke(123)

123

In [ ]:
passthrough.invoke({'a' : 1})

{'a': 1}

In [ ]:
def format_docs(docs):
    return '\n--\n'.join(
            f"[{d.metadata.get('category', '')}] {d.page_content}"
            for d in docs
    )

In [ ]:
docs = retriever.invoke('재택근무')

In [ ]:
docs

[Document(id='9355aa5d-922e-4318-b8c0-5407cf72633d', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.'),
 Document(id='9558cc81-2100-42af-afe2-7176e9d0dd71', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제4조 (교육)\n1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n3. 온라인 학습 플랫폼 이용료를 전액 지원한다.'),
 Document(id='490be4d7-2fe3-406a-b6aa-18e7172e902f', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.')]

In [ ]:
formatted = format_docs(docs)

In [ ]:
print(formatted)

[사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
--
[사내규정] 제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
--
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [ ]:
retriever_chain = retriever | format_docs
context_text = retriever_chain.invoke('재택근무')

In [ ]:
print(context_text)

[사내규정] 제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.
--
[사내규정] 제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
--
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.


In [ ]:
question = '재택근무 규정이 어떻게 되나요?'
docs = voctorstore.similarity_search(question, k=3)
context = '\n--\n'.join(
            f"[{d.metadata.get('category', '')}] {d.page_content}"
            for d in docs)
messages = [
    SystemMessage(),
    HumanMessage(),
    ...
]

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system" , "사내 도우미 챗봇입니다. 참고문서:\n{context}\n\n문서 기반으로 답변해주세요."),
    ("user", "{question}")
])

rag_chain = (  {"context" : retriever | format_docs,
                "question" : RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )

In [2]:
# k=1로 설정된 retriever를 생성합니다.
retriever_k1 = vectorstore.as_retriever(search_kwargs={'k':1})

# k=1 retriever를 사용하는 새로운 rag_chain을 생성합니다.
rag_chain_k1 = (  {"context" : retriever_k1 | format_docs,
                "question" : RunnablePassthrough()}
            | rag_prompt
            | llm
            | StrOutputParser()
            )

NameError: name 'vectorstore' is not defined

In [ ]:
answer = rag_chain.invoke('재택근무 규정이 어떻게 되나요?')
print(answer)

재택근무는 주  2회까지 가능합니다. 더 자세한 정보가 필요하시면 알려주세요!


In [ ]:
question_list = ['스마트홈 허브 초기 설정 방법', 'ai 산업 성장률', '교육비 지원 한도']
for q in question_list:
    print(f"Q : {q}")
    print(f"A : {rag_chain.invoke(q)}")
    print("======")

Q : 스마트홈 허브 초기 설정 방법
A : 스마트 홈 허브 v3.0의 초기 설정 방법은 다음과 같습니다.

1. 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
2. 모바일 앱을 설치하고 QR 코드를 스캔합니다.
3. 연동할 IoT 기기를 검색하고 등록합니다.

이 과정을 통해 스마트 홈 허브를 설정할 수 있습니다. 추가적인 질문이 있으시면 말씀해 주세요!
Q : ai 산업 성장률
A : 2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했습니다. 이 성장의 60%는 특히 생성형 AI 분야가 견인했습니다. 2025년에는 AI 산업이 약 7,000억 달러로 성장할 것으로 예상됩니다.
Q : 교육비 지원 한도
A : 교육비 지원 한도는 다음과 같습니다: 외부 컨퍼런스 참석비를 연 200만원까지 지원합니다. 또한, 온라인 학습 플랫폼 이용료는 전액 지원합니다.
